# Modeling

Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import joblib
from tqdm import tqdm
from sklearn.model_selection import TimeSeriesSplit

Load WTI Data

In [77]:
wti = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v3.parquet")

Load Tweet Data

In [78]:
tweets = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6.parquet")

In [79]:
tweets.shape

(15242, 53)

In [80]:
tweets.columns

Index(['timestamp_utc', 'clean_text', 'embedding', 'finbert_positive',
       'finbert_neutral', 'finbert_negative', 'finbert_sentiment_score',
       'finbert_entropy', 'finbert_confidence', 'financial_uncertainty_score',
       'financial_risk_sentiment', 'finbert_polarization', 'caps_ratio',
       'exclamation_count', 'exclamation_count_log', 'tweet_length',
       'tweet_length_log', 'market_aggression_index',
       'time_since_last_tweet_min', 'rolling_tweet_frequency_60m',
       'rolling_tweet_frequency_6h', 'rolling_tweet_frequency_24h',
       'tweet_burst_indicator', 'tweet_acceleration_6h',
       'sentiment_delta_vs_previous', 'rolling_sentiment_mean_60m',
       'rolling_sentiment_std_60m', 'is_trump_president', 'wti_bullish_score',
       'wti_bearish_score', 'energy_supply_score',
       'geopolitical_oil_risk_score', 'usd_strength_pressure_score',
       'fed_monetary_policy_score', 'risk_sentiment_score',
       'trump_energy_policy_score', 'trump_market_shock_langua

In [81]:
wti.shape

(4206118, 77)

In [82]:
wti.columns

Index(['timestamp_utc', 'open', 'high', 'low', 'close', 'was_missing',
       'log_close', 'ret_1', 'hl_range', 'oc_return', 'ret_5', 'ret_15',
       'ret_30', 'ret_60', 'ret_240', 'mom_5', 'mom_15', 'mom_60',
       'vol_close_5', 'vol_parkinson_5', 'vol_close_15', 'vol_parkinson_15',
       'vol_close_30', 'vol_parkinson_30', 'vol_close_60', 'vol_parkinson_60',
       'vol_close_240', 'vol_parkinson_240', 'vol_gk', 'vol_ratio_5_60',
       'sma_5', 'ema_5', 'dist_ema_5', 'sma_15', 'ema_15', 'dist_ema_15',
       'sma_60', 'ema_60', 'dist_ema_60', 'sma_240', 'ema_240', 'dist_ema_240',
       'hl_momentum', 'body', 'upper_wick', 'lower_wick', 'abs_ret',
       'vol_cluster', 'vol_5', 'jump', 'hour', 'dayofweek', 'hour_sin',
       'hour_cos', 'dow_sin', 'dow_cos', 'us_session', 'vol_rank',
       'vol_regime', 'trend_60', 'trend_regime', 'y_up_2', 'y_down_2',
       'y_up_5', 'y_down_5', 'y_up_15', 'y_down_15', 'y_up_30', 'y_down_30',
       'y_up_60', 'y_down_60', 'y_up_240', 'y_down

## Train-Validation-Test-Split

In [83]:
tweets_pre2020 = tweets[tweets['timestamp_utc'] < '2020-01-01'] # wird für training und validation verwendet
tweets_post2020 = tweets[tweets['timestamp_utc'] >= '2020-01-01'] # als test set für die out of sample performance

## ML-Pipeline

### 1. Config

In [13]:
PCA_COMPONENTS = 20
N_SPLITS = 5
PURGE_MINUTES = 1440

TIME_WTI = "timestamp_utc"
TIME_TWEET = "timestamp_utc"

### 2. Embedding Parser

In [14]:
def parse_embedding(x):
    return np.fromstring(x.strip("[]"), sep=" ")

### 3. Tweet Preprocessing + Embeddings

In [15]:
def prepare_tweets(df):

    df = df.copy()

    df[TIME_TWEET] = pd.to_datetime(df[TIME_TWEET], utc=True)

    df["embedding_vec"] = df["embedding"].apply(parse_embedding)

    emb = np.vstack(df["embedding_vec"].values)

    emb_cols = [f"emb_{i}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)

    df = pd.concat([df.reset_index(drop=True), emb_df], axis=1)

    return df

### 4. Tweet Aggregation

Notiz: Hier fehlen noch die aggregation für die restlichen spalten! also unvollständig

In [45]:
def aggregate_tweets(df):

    df = df.copy()

    df["minute"] = df[TIME_TWEET].dt.floor("min")

    emb_cols = [c for c in df.columns if c.startswith("emb_")]

    agg_dict = {

        # =========================
        # FinBERT
        # =========================
        "finbert_positive": "mean",
        "finbert_neutral": "mean",
        "finbert_negative": "mean",
        "finbert_sentiment_score": "mean",
        "finbert_entropy": "mean",
        "finbert_confidence": "max",
        "finbert_polarization": "max",

        # =========================
        # Financial NLP Scores
        # =========================
        "financial_uncertainty_score": "max",
        "financial_risk_sentiment": "mean",

        # =========================
        # Text Style
        # =========================
        "caps_ratio": "mean",
        "exclamation_count": "max",
        "exclamation_count_log": "max",

        "tweet_length": "mean",
        "tweet_length_log": "mean",

        "market_aggression_index": "max",

        # =========================
        # Tweet Dynamics
        # =========================
        "time_since_last_tweet_min": "min",

        "rolling_tweet_frequency_60m": "max",
        "rolling_tweet_frequency_6h": "max",
        "rolling_tweet_frequency_24h": "max",

        "tweet_burst_indicator": "max",
        "tweet_acceleration_6h": "max",

        # =========================
        # Sentiment Dynamics
        # =========================
        "sentiment_delta_vs_previous": "mean",
        "rolling_sentiment_mean_60m": "mean",
        "rolling_sentiment_std_60m": "mean",

        # =========================
        # Presidency Regime
        # =========================
        "is_trump_president": "max",

        # =========================
        # Oil / Macro Scores
        # =========================
        "wti_bullish_score": "max",
        "wti_bearish_score": "max",

        "energy_supply_score": "max",
        "geopolitical_oil_risk_score": "max",
        "usd_strength_pressure_score": "max",
        "fed_monetary_policy_score": "max",
        "risk_sentiment_score": "max",

        "trump_energy_policy_score": "max",
        "trump_market_shock_language_score": "max",
        "policy_shock_score": "max",

        # =========================
        # Topic Metrics
        # =========================
        "topic_concentration": "mean",

        "topic_energy_supply": "max",
        "topic_fed_monetary_policy": "max",
        "topic_geopolitical_oil_risk": "max",
        "topic_policy_shock": "max",
        "topic_risk_sentiment": "max",
        "topic_trump_energy_policy": "max",
        "topic_trump_market_shock_language": "max",
        "topic_usd_strength_pressure": "max",
        "topic_wti_bearish": "max",
        "topic_wti_bullish": "max",

        # =========================
        # Semantic Change
        # =========================
        "semantic_global_novelty": "max",
        "semantic_local_change": "max",
        "semantic_rolling_novelty": "max",
    }

    # =========================
    # Embeddings
    # =========================
    for c in emb_cols:
        agg_dict[c] = "mean"

    agg = (
        df.groupby("minute")
          .agg(agg_dict)
          .reset_index()
    )

    return agg

In [ ]:
# def aggregate_tweets(df):

#     df = df.copy()

#     df["minute"] = df[TIME_TWEET].dt.floor("min")

#     emb_cols = [c for c in df.columns if c.startswith("emb_")]

#     agg_dict = {
#         "finbert_sentiment_score": ["mean", "std"],
#         "financial_uncertainty_score": ["mean", "max"],
#         "tweet_length": ["mean", "sum"],
#         "exclamation_count": ["mean"]
#     }

#     for c in emb_cols:
#         agg_dict[c] = ["mean", "max", "min"]

#     agg = df.groupby("minute").agg(agg_dict)

#     agg.columns = ["_".join(col) for col in agg.columns]
#     agg = agg.reset_index()

#     return agg

### 5. Leakage-Free WTI Mapping

In [17]:
def map_to_wti(tweet_agg, wti):

    tweet_agg = tweet_agg.copy()
    wti = wti.copy()

    tweet_agg[TIME_TWEET] = pd.to_datetime(tweet_agg["minute"], utc=True)
    wti[TIME_WTI] = pd.to_datetime(wti[TIME_WTI], utc=True)

    # KEY RULE:
    # tweet -> previous completed minute candle
    tweet_agg["wti_time"] = tweet_agg["minute"] - pd.Timedelta(minutes=1)

    df = tweet_agg.merge(
        wti,
        left_on="wti_time",
        right_on=TIME_WTI,
        how="left"
    )

    df = df.dropna(subset=["open"])

    return df

### 6. Feature / Target Split

In [18]:
TARGETS = [c for c in wti.columns if c.startswith("y_")]

def split_xy(df):

    y = df[TARGETS]
    X = df.drop(columns=TARGETS)

    return X, y

### 7. Purged CV

## nutze lieber time series split - ist eine getestete bibliothek

In [ ]:
# class PurgedCV:

#     def __init__(self, n_splits=5, purge=1440):
#         self.n_splits = n_splits
#         self.purge = purge

#     def split(self, X):

#         n = len(X)
#         idx = np.arange(n)
#         fold = n // self.n_splits

#         for i in range(self.n_splits):

#             val_start = i * fold
#             val_end = n if i == self.n_splits - 1 else (i + 1) * fold

#             val_idx = idx[val_start:val_end]

#             train_mask = np.ones(n, dtype=bool)

#             train_mask[
#                 max(0, val_start - self.purge):
#                 min(n, val_end + self.purge)
#             ] = False

#             train_idx = idx[train_mask]

#             yield train_idx, val_idx

### 8. Models

In [19]:
MODELS = {
    "logreg": LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=5000,
        class_weight="balanced"
    ),

    "rf": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42
    ),

    "mlp": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=300
    )
}

### 9. Evaluation

In [20]:
def evaluate(y_true, pred, prob):

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "f1": f1_score(y_true, pred),
        "acc": accuracy_score(y_true, pred)
    }

In [21]:
def clean_features(df):

    # print(df.isna().sum()[df.isna().sum() > 0])

    # nur numerische Features
    df = df.select_dtypes(include=[np.number])

    # df = df.dropna()

    return df


## Start: Durchführung einiger Tests und Korrekturen

In [103]:
# import pandas as pd

def show_class_distribution(df, horizons=[2,5,15,30,60,240,720,1440]):
    results = []

    for h in horizons:
        up_col = f"y_up_{h}"
        down_col = f"y_down_{h}"

        if up_col in df.columns:
            up_counts = df[up_col].value_counts(normalize=True).sort_index()
            results.append({
                "horizon": h,
                "target": up_col,
                "class_0": up_counts.get(0, 0.0),
                "class_1": up_counts.get(1, 0.0)
            })

        if down_col in df.columns:
            down_counts = df[down_col].value_counts(normalize=True).sort_index()
            results.append({
                "horizon": h,
                "target": down_col,
                "class_0": down_counts.get(0, 0.0),
                "class_1": down_counts.get(1, 0.0)
            })

    result_df = pd.DataFrame(results)
    return result_df


# Nutzung:
dist = show_class_distribution(df)
print(dist)

    horizon       target   class_0   class_1
0         2       y_up_2  0.790452  0.209548
1         2     y_down_2  0.786492  0.213508
2         5       y_up_5  0.746563  0.253437
3         5     y_down_5  0.746013  0.253987
4        15      y_up_15  0.701023  0.298977
5        15    y_down_15  0.703773  0.296227
6        30      y_up_30  0.682103  0.317897
7        30    y_down_30  0.688923  0.311077
8        60      y_up_60  0.660103  0.339897
9        60    y_down_60  0.674733  0.325267
10      240     y_up_240  0.610164  0.389836
11      240   y_down_240  0.668793  0.331207
12      720     y_up_720  0.583324  0.416676
13      720   y_down_720  0.623804  0.376196
14     1440    y_up_1440  0.556924  0.443076
15     1440  y_down_1440  0.606644  0.393356


In [118]:
baseline_cols = [
    'open', 'high', 'low', 'close', 'was_missing',
    'log_close', 'ret_1', 'hl_range', 'oc_return', 'ret_5', 'ret_15',
    'ret_30', 'ret_60', 'ret_240', 'mom_5', 'mom_15', 'mom_60',
    'vol_close_5', 'vol_parkinson_5', 'vol_close_15', 'vol_parkinson_15',
    'vol_close_30', 'vol_parkinson_30', 'vol_close_60', 'vol_parkinson_60',
    'vol_close_240', 'vol_parkinson_240', 'vol_gk', 'vol_ratio_5_60',
    'sma_5', 'ema_5', 'dist_ema_5', 'sma_15', 'ema_15', 'dist_ema_15',
    'sma_60', 'ema_60', 'dist_ema_60', 'sma_240', 'ema_240', 'dist_ema_240',
    'hl_momentum', 'body', 'upper_wick', 'lower_wick', 'abs_ret',
    'vol_cluster', 'vol_5', 'jump', 'hour', 'dayofweek', 'hour_sin',
    'hour_cos', 'dow_sin', 'dow_cos', 'us_session', 'vol_rank',
    'vol_regime', 'trend_60', 'trend_regime'
]

feature_sets = {
    "baseline": clean_features(
        X[baseline_cols].copy()
    ),
    "enhanced": clean_features(X)
}


In [121]:
feature_sets["enhanced"].columns

Index(['finbert_positive', 'finbert_neutral', 'finbert_negative',
       'finbert_sentiment_score', 'finbert_entropy', 'finbert_confidence',
       'finbert_polarization', 'financial_uncertainty_score',
       'financial_risk_sentiment', 'caps_ratio',
       ...
       'dayofweek', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'us_session',
       'vol_rank', 'vol_regime', 'trend_60', 'trend_regime'],
      dtype='str', length=493)

## Ende

### 10. Core Training Loop

To Dos:
- feature selection
- feature importance + diagnostik fehlt
- hyperparameter tuning fehlt
- modelle müssen angepasst werden - MLP bewusster konfigurieren etc., early stopping etc. pp
- overfitting detecten und beheben
- andere metriken nehmen
- in dataframe wegschreiben und im nachgang analysierenm, auch feature importance, residen, testing etc.
- Visualisierungen bei timerseries split, learning curves etc.
- Residuenanalye + Diagnostik!
- ggf das finale model nochmal auf kompletten Daten trainieren


# Aktueller Ansatz:

Config

In [ ]:
CONFIG = {
    "outer_splits": 5,
    "inner_splits": 3,
    "gap": 30,

    "targets": ["target_30min"],
    "feature_sets": ["baseline", "enhanced"],
    "models": ["logreg", "xgboost", "mlp"],

    "optuna_trials": 30,

    "top_k_features": 0.7,
}

Data Layer

In [ ]:
def build_dataset(wti_df, tweets_df):

    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)

    return df, X, y

Feature Sets

In [ ]:
def get_feature_sets(X):

    baseline_cols = [
        'open','high','low','close','was_missing',
        'log_close','ret_1','hl_range','oc_return',
        'ret_5','ret_15','ret_30','ret_60',
        'mom_5','mom_15','mom_60',
        'vol_close_5','vol_parkinson_5',
        'sma_5','ema_5','dist_ema_5',
        'hour','dayofweek','hour_sin','hour_cos'
    ]

    return {
        "baseline": clean_features(X[baseline_cols].copy()),
        "enhanced": clean_features(X.copy())
    }

CV Splits

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def outer_cv():
    return TimeSeriesSplit(
        n_splits=CONFIG["outer_splits"],
        gap=CONFIG["gap"]
    )

def inner_cv():
    return TimeSeriesSplit(
        n_splits=CONFIG["inner_splits"],
        gap=CONFIG["gap"]
    )

Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier


def build_model(name, params):

    if name == "logreg":
        return LogisticRegression(
            penalty="l1",
            solver="saga",
            max_iter=5000,
            class_weight="balanced",
            **params
        )

    if name == "xgboost":
        return XGBClassifier(
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            **params
        )

    if name == "mlp":
        return MLPClassifier(
            max_iter=300,
            early_stopping=True,
            **params
        )

Pipeline Buidling

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

Feature Selector - Tree Model

In [ ]:
class TreeFeatureSelector(BaseEstimator, TransformerMixin):

    def __init__(self, model, top_k=0.7):
        self.model = model
        self.top_k = top_k

    def fit(self, X, y):

        self.model.fit(X, y)
        importances = self.model.feature_importances_

        k = int(len(importances) * self.top_k)
        self.selected_idx_ = np.argsort(importances)[-k:]

        return self

    def transform(self, X):
        return X.iloc[:, self.selected_idx_]

Pipeline Factory

In [ ]:
def make_pipeline(model_name, params):

    model = build_model(model_name, params)

    steps = []

    # scaling only for linear / neural
    if model_name in ["logreg", "mlp"]:
        steps.append(("scaler", StandardScaler()))

    # feature selection only for tree models
    if model_name == "xgboost":
        steps.append(("selector", TreeFeatureSelector(model)))

    steps.append(("model", model))

    return Pipeline(steps)

Optuna (inner CV only)

In [ ]:
import optuna
from sklearn.metrics import roc_auc_score


def tune_model(model_name, X_train, y_train, inner_cv):

    def objective(trial):

        if model_name == "logreg":
            params = {
                "C": trial.suggest_float("C", 1e-3, 10, log=True)
            }

        elif model_name == "xgboost":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 600),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("lr", 0.01, 0.2, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample", 0.6, 1.0),
            }

        elif model_name == "mlp":
            params = {
                "hidden_layer_sizes": (
                    trial.suggest_int("h1", 64, 256),
                    trial.suggest_int("h2", 32, 128),
                ),
                "alpha": trial.suggest_float("alpha", 1e-5, 1e-2, log=True)
            }

        scores = []

        for tr, va in inner_cv.split(X_train):

            X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
            y_tr, y_va = y_train.iloc[tr], y_train.iloc[va]

            pipe = make_pipeline(model_name, params)
            pipe.fit(X_tr, y_tr)

            prob = pipe.predict_proba(X_va)[:, 1]
            scores.append(roc_auc_score(y_va, prob))

        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=CONFIG["optuna_trials"])

    return study.best_params

Outer Fold Training (Unbiased)

In [ ]:
def run_fold(model_name, best_params, X_train, X_val, y_train, y_val):

    pipe = make_pipeline(model_name, best_params)

    pipe.fit(X_train, y_train)

    prob = pipe.predict_proba(X_val)[:, 1]
    pred = (prob > 0.5).astype(int)

    return pipe, pred, prob

Experiment Loop (Full nested CV)

In [ ]:
def run_experiment(wti_df, tweets_df):

    df, X, y = build_dataset(wti_df, tweets_df)
    feature_sets = get_feature_sets(X)

    outer = outer_cv()
    inner = inner_cv()

    results = []
    predictions = []
    models = {}

    for target in CONFIG["targets"]:

        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():

            for model_name in CONFIG["models"]:

                # -------------------------
                # INNER CV (TUNING)
                # -------------------------
                best_params = tune_model(
                    model_name,
                    X_fs,
                    y_t,
                    inner
                )

                # -------------------------
                # OUTER CV (EVALUATION)
                # -------------------------
                for fold, (tr, va) in enumerate(outer.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    model, pred, prob = run_fold(
                        model_name,
                        best_params,
                        X_train,
                        X_val,
                        y_train,
                        y_val
                    )

                    scores = evaluate(y_val, pred, prob)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        **scores
                    })

                    predictions.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        "y_true": y_val.values,
                        "y_pred": pred,
                        "y_prob": prob,
                        "index": y_val.index
                    })

                    models[(target, fs_name, model_name, fold)] = model

    return results, predictions, models

### Post Analysis Module

Model Ranking

In [ ]:
def rank_models(results_df):
    return results_df.groupby(["model", "feature_set"]).mean()

Stability Analysis

In [ ]:
def fold_stability(results_df):
    return results_df.groupby("model")["roc_auc"].std()

Prediction Drift

In [ ]:
def plot_time_performance(predictions_df):

    df = pd.DataFrame(predictions_df)
    df["correct"] = df["y_pred"] == df["y_true"]

    return df.groupby("fold")["correct"].mean()

SHAP (after training only)

In [ ]:
import shap

def shap_analysis(model, X):

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    shap.summary_plot(shap_values, X)

# 2 Pipeline Ansätze

# laut Chat ist das hier die beste Pipeline!

Aber aktuelle Kritk: Das Hyperparameter-Tuning erfolgt einmal auf dem gesamten Datensatz vor dem Outer-CV.

In [ ]:
# =========================
# CONFIG
# =========================

CONFIG = {
    "n_splits_outer": 5,
    "n_splits_inner": 3,
    "gap": 30,
    "targets": ["target_30min"],
    "feature_sets": ["baseline", "enhanced"],
    "models": ["logreg", "xgboost", "mlp"],
    "optuna_trials": 30,
    "use_feature_selection": True,
}


# =========================
# DATA LAYER
# =========================

def build_dataset(wti_df, tweets_df):
    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)

    return df, X, y


# =========================
# FEATURE SETS
# =========================

def get_feature_sets(X):
    baseline_cols = [
        'open','high','low','close','was_missing',
        'log_close','ret_1','hl_range','oc_return',
        'ret_5','ret_15','ret_30','ret_60',
        'mom_5','mom_15','mom_60',
        'vol_close_5','vol_parkinson_5',
        'sma_5','ema_5','dist_ema_5',
        'hour','dayofweek','hour_sin','hour_cos'
    ]

    return {
        "baseline": clean_features(X[baseline_cols].copy()),
        "enhanced": clean_features(X.copy())
    }


# =========================
# CV
# =========================

from sklearn.model_selection import TimeSeriesSplit

def get_outer_cv():
    return TimeSeriesSplit(n_splits=CONFIG["n_splits_outer"], gap=CONFIG["gap"])

def get_inner_cv():
    return TimeSeriesSplit(n_splits=CONFIG["n_splits_inner"], gap=CONFIG["gap"])


# =========================
# MODEL FACTORY
# =========================

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

def build_model(name, params):

    if name == "logreg":
        return LogisticRegression(
            penalty="l1",
            solver="saga",
            max_iter=5000,
            class_weight="balanced",
            **params
        )

    if name == "xgboost":
        return XGBClassifier(
            eval_metric="logloss",
            tree_method="hist",
            **params
        )

    if name == "mlp":
        return MLPClassifier(
            max_iter=300,
            early_stopping=True,
            **params
        )


# =========================
# PREPROCESSING (FIXED)
# =========================

from sklearn.preprocessing import StandardScaler
import pandas as pd

def scale_data(model_name, X_train, X_val):

    if model_name in ["logreg", "mlp"]:
        scaler = StandardScaler()

        X_train_s = pd.DataFrame(
            scaler.fit_transform(X_train),
            columns=X_train.columns,
            index=X_train.index
        )

        X_val_s = pd.DataFrame(
            scaler.transform(X_val),
            columns=X_val.columns,
            index=X_val.index
        )

        return X_train_s, X_val_s, scaler

    return X_train, X_val, None


# =========================
# FEATURE SELECTION
# =========================

from sklearn.feature_selection import mutual_info_classif
import numpy as np

def feature_selection(model_name, model, X_train, y_train):

    if not CONFIG["use_feature_selection"]:
        return X_train.columns

    if model_name == "logreg":
        return X_train.columns

    if model_name == "xgboost":
        model.fit(X_train, y_train)
        importances = model.feature_importances_

        top_k = int(len(importances) * 0.7)
        idx = np.argsort(importances)[-top_k:]

        return X_train.columns[idx]

    if model_name == "mlp":
        mi = mutual_info_classif(X_train, y_train)
        top_k = int(len(mi) * 0.7)
        idx = np.argsort(mi)[-top_k:]

        return X_train.columns[idx]


# =========================
# FOLD TRAINING
# =========================

def run_fold(model_name, model, X_train, X_val, y_train, y_val):

    X_train_s, X_val_s, scaler = scale_data(model_name, X_train, X_val)

    selected_cols = feature_selection(
        model_name,
        model,
        X_train_s,
        y_train
    )

    X_train_f = X_train_s[selected_cols]
    X_val_f = X_val_s[selected_cols]

    model.fit(X_train_f, y_train)

    prob = model.predict_proba(X_val_f)[:, 1]
    pred = (prob > 0.5).astype(int)

    return model, scaler, selected_cols, pred, prob


# =========================
# OPTUNA TUNING (INNER CV)
# =========================

import optuna
from sklearn.metrics import roc_auc_score

def tune_model(model_name, X_train, y_train, inner_cv):

    def objective(trial):

        if model_name == "logreg":
            params = {
                "C": trial.suggest_float("C", 1e-3, 10, log=True),
            }

        elif model_name == "xgboost":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 500),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("lr", 0.01, 0.3, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            }

        elif model_name == "mlp":
            params = {
                "hidden_layer_sizes": (
                    trial.suggest_int("h1", 64, 256),
                    trial.suggest_int("h2", 32, 128),
                ),
                "alpha": trial.suggest_float("alpha", 1e-5, 1e-2, log=True),
            }

        scores = []

        for tr, va in inner_cv.split(X_train):

            X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
            y_tr, y_va = y_train.iloc[tr], y_train.iloc[va]

            model = build_model(model_name, params)
            model.fit(X_tr, y_tr)

            prob = model.predict_proba(X_va)[:, 1]
            scores.append(roc_auc_score(y_va, prob))

        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=CONFIG["optuna_trials"])

    return study.best_params


# =========================
# EXPERIMENT LOOP (OUTER CV)
# =========================

def run_experiment(wti_df, tweets_df, config):

    df, X, y = build_dataset(wti_df, tweets_df)
    feature_sets = get_feature_sets(X)

    outer_cv = get_outer_cv()
    inner_cv = get_inner_cv()

    results = []
    predictions_store = []
    models_store = {}

    for target in config["targets"]:
        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():

            for model_name in config["models"]:

                best_params = tune_model(model_name, X_fs, y_t, inner_cv)

                for fold, (tr, va) in enumerate(outer_cv.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    model = build_model(model_name, best_params)

                    model, scaler, feats, pred, prob = run_fold(
                        model_name,
                        model,
                        X_train,
                        X_val,
                        y_train,
                        y_val
                    )

                    scores = evaluate(y_val, pred, prob)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        **scores
                    })

                    predictions_store.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        "y_true": y_val.values,
                        "y_pred": pred,
                        "y_prob": prob,
                        "index": y_val.index
                    })

                    models_store[(target, fs_name, model_name, fold)] = {
                        "model": model,
                        "scaler": scaler,
                        "features": feats
                    }

    return pd.DataFrame(results), predictions_store, models_store


# =========================
# ANALYSIS HELPERS
# =========================

def rank_models(results_df):
    return results_df.groupby(["model", "feature_set"]).mean()


import shap

def shap_analysis(model, X):

    if hasattr(model, "feature_importances_"):
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
        shap.summary_plot(shap_values, X)


def plot_performance(predictions):

    df = pd.DataFrame(predictions)
    df["correct"] = df["y_pred"] == df["y_true"]

    return df.groupby("fold")["correct"].mean()

In [ ]:
# =========================
# CONFIG
# =========================

CONFIG = {
    "n_splits": 5,
    "gap": 30,
    "targets": ["target_30min"],
    "feature_sets": ["baseline", "enhanced"],
    "models": ["logreg", "xgboost", "mlp"],
    "optuna_trials": 30,
    "use_feature_selection": True,
}


# =========================
# DATA LAYER
# =========================

def build_dataset(wti_df, tweets_df):
    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)

    return df, X, y


# =========================
# FEATURE SETS
# =========================

def get_feature_sets(X):
    baseline_cols = [
        'open','high','low','close','was_missing',
        'log_close','ret_1','hl_range','oc_return',
        'ret_5','ret_15','ret_30','ret_60',
        'mom_5','mom_15','mom_60',
        'vol_close_5','vol_parkinson_5',
        'sma_5','ema_5','dist_ema_5',
        'hour','dayofweek','hour_sin','hour_cos'
    ]

    return {
        "baseline": clean_features(X[baseline_cols].copy()),
        "enhanced": clean_features(X.copy())
    }


# =========================
# CV
# =========================

def get_cv():
    return TimeSeriesSplit(
        n_splits=5,
        gap=30
    )


# =========================
# MODEL FACTORY
# =========================

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

def build_model(name, params):
    if name == "logreg":
        return LogisticRegression(
            penalty="l1",
            solver="saga",
            max_iter=5000,
            class_weight="balanced",
            **params
        )

    if name == "xgboost":
        return XGBClassifier(
            eval_metric="logloss",
            tree_method="hist",
            **params
        )

    if name == "mlp":
        return MLPClassifier(
            max_iter=300,
            early_stopping=True,
            **params
        )


# =========================
# FEATURE SELECTION
# =========================

def feature_selection(model_name, model, X_train, y_train):
    if model_name == "logreg":
        return X_train.columns

    if model_name == "xgboost":
        model.fit(X_train, y_train)
        importances = model.feature_importances_

        top_k = int(len(importances) * 0.7)
        idx = np.argsort(importances)[-top_k:]

        return X_train.columns[idx]

    if model_name == "mlp":
        from sklearn.feature_selection import mutual_info_classif

        mi = mutual_info_classif(X_train, y_train)
        top_k = int(len(mi) * 0.7)
        idx = np.argsort(mi)[-top_k:]

        return X_train.columns[idx]


# =========================
# FOLD TRAINING
# =========================

def run_fold(model_name, model, X_train, X_val, y_train, y_val):

    scaler = StandardScaler()

    X_train_s = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_val_s = pd.DataFrame(
        scaler.transform(X_val),
        columns=X_val.columns,
        index=X_val.index
    )

    selected_cols = feature_selection(
        model_name,
        model,
        X_train_s,
        y_train
    )

    X_train_f = X_train_s[selected_cols]
    X_val_f = X_val_s[selected_cols]

    model.fit(X_train_f, y_train)

    prob = model.predict_proba(X_val_f)[:, 1]
    pred = (prob > 0.5).astype(int)

    return model, scaler, selected_cols, pred, prob


# =========================
# OPTUNA TUNING
# =========================

import optuna

def tune_model(model_name, X_train, y_train, cv):

    def objective(trial):

        if model_name == "logreg":
            params = {
                "C": trial.suggest_loguniform("C", 1e-3, 10),
            }

        elif model_name == "xgboost":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 500),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_loguniform("lr", 0.01, 0.3),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            }

        elif model_name == "mlp":
            params = {
                "hidden_layer_sizes": (
                    trial.suggest_int("h1", 64, 256),
                    trial.suggest_int("h2", 32, 128)
                ),
                "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-2),
            }

        scores = []

        for tr, va in cv.split(X_train):

            X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
            y_tr, y_va = y_train.iloc[tr], y_train.iloc[va]

            model = build_model(model_name, params)

            model.fit(X_tr, y_tr)
            prob = model.predict_proba(X_va)[:, 1]

            score = roc_auc_score(y_va, prob)
            scores.append(score)

        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)

    return study.best_params


# =========================
# MAIN EXPERIMENT LOOP
# =========================

def run_experiment(wti_df, tweets_df, config):

    df, X, y = build_dataset(wti_df, tweets_df)
    feature_sets = get_feature_sets(X)
    cv = get_cv()

    results = []
    predictions_store = []
    models_store = {}

    for target in config["targets"]:
        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():

            for model_name in config["models"]:

                best_params = tune_model(model_name, X_fs, y_t, cv)

                for fold, (tr, va) in enumerate(cv.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    model = build_model(model_name, best_params)

                    model, scaler, feats, pred, prob = run_fold(
                        model_name,
                        model,
                        X_train,
                        X_val,
                        y_train,
                        y_val
                    )

                    scores = evaluate(y_val, pred, prob)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        **scores
                    })

                    predictions_store.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        "y_true": y_val.values,
                        "y_pred": pred,
                        "y_prob": prob,
                        "index": y_val.index
                    })

                    models_store[(target, fs_name, model_name, fold)] = {
                        "model": model,
                        "scaler": scaler,
                        "features": feats
                    }

    return pd.DataFrame(results), predictions_store, models_store


# =========================
# ANALYSIS HELPERS
# =========================

def rank_models(results_df):
    return results_df.groupby(["model", "feature_set"]).mean()


import shap

def shap_analysis(model, X):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    shap.summary_plot(shap_values, X)


def plot_performance(predictions):
    df = pd.DataFrame(predictions)
    df["correct"] = df["y_pred"] == df["y_true"]
    return df.groupby("fold")["correct"].mean()

### Alternativer Ansatz:

In [ ]:
def run_pipeline(wti_df, tweets_df):

    results = []
    models_store = {}

    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)
    
    # std_cols = [c for c in X.columns if c.endswith("_std")]
    # # print(std_cols)
    # for c in std_cols:
    #     X[c] = X[c].fillna(0)

    # feature_sets = {
    #     "baseline": clean_features(
    #         X.drop(columns=[c for c in X.columns if c.startswith("emb_")])
    #     ),
    #     "enhanced": clean_features(X)
    # }

    baseline_cols = [
        'open', 'high', 'low', 'close', 'was_missing',
        'log_close', 'ret_1', 'hl_range', 'oc_return', 'ret_5', 'ret_15',
        'ret_30', 'ret_60', 'ret_240', 'mom_5', 'mom_15', 'mom_60',
        'vol_close_5', 'vol_parkinson_5', 'vol_close_15', 'vol_parkinson_15',
        'vol_close_30', 'vol_parkinson_30', 'vol_close_60', 'vol_parkinson_60',
        'vol_close_240', 'vol_parkinson_240', 'vol_gk', 'vol_ratio_5_60',
        'sma_5', 'ema_5', 'dist_ema_5', 'sma_15', 'ema_15', 'dist_ema_15',
        'sma_60', 'ema_60', 'dist_ema_60', 'sma_240', 'ema_240', 'dist_ema_240',
        'hl_momentum', 'body', 'upper_wick', 'lower_wick', 'abs_ret',
        'vol_cluster', 'vol_5', 'jump', 'hour', 'dayofweek', 'hour_sin',
        'hour_cos', 'dow_sin', 'dow_cos', 'us_session', 'vol_rank',
        'vol_regime', 'trend_60', 'trend_regime'
    ]

    feature_sets = {
        "baseline": clean_features(
            X[baseline_cols].copy()
        ),
        "enhanced": clean_features(X)
    }


    # cv = PurgedCV(n_splits=N_SPLITS, purge=PURGE_MINUTES)
    cv = TimeSeriesSplit(
        n_splits=N_SPLITS,
        gap=PURGE_MINUTES
    )

    for target in TARGETS:

        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():

            # Embedding-Spalten einmal pro Feature Set bestimmen
            emb_cols = [c for c in X_fs.columns if c.startswith("emb_")]
            non_emb_cols = [c for c in X_fs.columns if c not in emb_cols]
            
            has_embeddings = len(emb_cols) > 0            

            for model_name, model in MODELS.items():

                for fold, (tr, va) in enumerate(cv.split(X_fs)):

                    X_train_df, X_val_df = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    # -------------------------
                    # 1) SCALE (auf allen Features)
                    # -------------------------
                    scaler = StandardScaler()
                    X_train_scaled = pd.DataFrame(
                        scaler.fit_transform(X_train_df),
                        columns=X_fs.columns,
                        index=X_train_df.index
                    )
                    X_val_scaled = pd.DataFrame(
                        scaler.transform(X_val_df),
                        columns=X_fs.columns,
                        index=X_val_df.index
                    )

                    if has_embeddings:

                        # -------------------------
                        # 2) PCA NUR auf Embeddings
                        # -------------------------
                        pca = PCA(n_components=50, random_state=42)

                        X_train_emb = X_train_scaled[emb_cols]
                        X_val_emb = X_val_scaled[emb_cols]

                        X_train_pca = pca.fit_transform(X_train_emb)
                        X_val_pca = pca.transform(X_val_emb)

                        # PCA-Spaltennamen
                        pca_cols = [f"emb_pca_{i}" for i in range(X_train_pca.shape[1])]

                        X_train_pca_df = pd.DataFrame(X_train_pca, columns=pca_cols, index=X_train_df.index)
                        X_val_pca_df = pd.DataFrame(X_val_pca, columns=pca_cols, index=X_val_df.index)

                        # -------------------------
                        # 3) Embeddings droppen + PCA ersetzen
                        # -------------------------
                        X_train_final = pd.concat(
                            [X_train_scaled[non_emb_cols], X_train_pca_df],
                            axis=1
                        )

                        X_val_final = pd.concat(
                            [X_val_scaled[non_emb_cols], X_val_pca_df],
                            axis=1
                        )

                    else:
                        # baseline case: keine embeddings → kein PCA
                        X_train_final = X_train_scaled.copy()
                        X_val_final = X_val_scaled.copy()

                    # 4) Model Training

                    model.fit(X_train_final, y_train)

                    prob = model.predict_proba(X_val_final)[:, 1]
                    pred = (prob > 0.5).astype(int)

                    scores = evaluate(y_val, pred, prob)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        **scores
                    })

                    models_store[(target, fs_name, model_name, fold)] = model

    return pd.DataFrame(results), models_store

In [126]:
print(tweets_agg.dtypes.to_string())

minute                               datetime64[us, UTC]
finbert_positive                                 float64
finbert_neutral                                  float64
finbert_negative                                 float64
finbert_sentiment_score                          float64
finbert_entropy                                  float64
finbert_confidence                               float64
finbert_polarization                             float64
financial_uncertainty_score                      float64
financial_risk_sentiment                         float64
caps_ratio                                       float64
exclamation_count                                  int64
exclamation_count_log                            float64
tweet_length                                     float64
tweet_length_log                                 float64
market_aggression_index                          float64
time_since_last_tweet_min                        float64
rolling_tweet_frequency_60m    

### Bisher verwendeter Training Loop

In [ ]:
def run_pipeline(wti_df, tweets_df):

    results = []
    models_store = {}

    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)
    
    # Beginn von mir ergänzer Code
    std_cols = [c for c in X.columns if c.endswith("_std")]
    # print(std_cols)
    for c in std_cols:
        X[c] = X[c].fillna(0)
    # Ende von mir ergänzer Code

    # feature_sets = {
    #     "baseline": X.drop(columns=[c for c in X.columns if c.startswith("emb_")]),
    #     "enhanced": X
    # }
    feature_sets = {
        "baseline": clean_features(
            X.drop(columns=[c for c in X.columns if c.startswith("emb_")])
        ),
        "enhanced": clean_features(X)
    }
    # cv = PurgedCV(n_splits=N_SPLITS, purge=PURGE_MINUTES)
    cv = TimeSeriesSplit(
        n_splits=N_SPLITS,
        gap=PURGE_MINUTES
    )

    for target in TARGETS:

        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():

            for model_name, model in MODELS.items():

                for fold, (tr, va) in enumerate(cv.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    # print(X_train.columns)
                    # print(X_train.dtypes)
                    # X_train.drop("minute", axis=1, inplace=True)
                    # X_val.drop("minute", axis = 1, inplace = True)

                    scaler = StandardScaler()
                    X_train = scaler.fit_transform(X_train)
                    X_val = scaler.transform(X_val)

                    model.fit(X_train, y_train)

                    prob = model.predict_proba(X_val)[:, 1]
                    pred = (prob > 0.5).astype(int)

                    scores = evaluate(y_val, pred, prob)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        **scores
                    })

                    models_store[(target, fs_name, model_name, fold)] = model

    return pd.DataFrame(results), models_store

hier fehlt noch die training scores auszuwerten, nicht nur die validations damit man overfitting etc. erkennen kann

### 11. Best Model Export

In [56]:
def save_best(results, models_store):

    best = results.sort_values("roc_auc", ascending=False).iloc[0]

    key = (
        best.target,
        best.feature_set,
        best.model,
        best.fold
    )

    joblib.dump(models_store[key], "best_model.pkl")

    return best

In [74]:
std_cols = [c for c in tweets_pre2020.columns if c.endswith("_std")]
print(std_cols)

[]


### 12. RUN

In [77]:
results, models = run_pipeline(wti, tweets_pre2020)

best_model_info = save_best(results, models)

print(best_model_info)

C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_17404\1248267005.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg = agg.reset_index()


Series([], dtype: int64)
Series([], dtype: int64)


c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use 

target         y_down_2
feature_set    baseline
model                rf
fold                  4
roc_auc        0.600463
f1             0.095745
acc            0.702016
Name: 39, dtype: object


In [84]:
results.to_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\results_v1.parquet", index=False)

In [89]:
len(models)

480